# Lab 2: Feature Engineering

**Goal**: Beat the baseline heuristic model by engineering 3 new features using F1 domain knowledge. Check for data leakage to ensure `IsTop10` predictions remain strictly driven by pre-race variables.

### Required Features:
1. **Lag feature**: `prev_race_position`
2. **Rolling aggregate**: `avg_position_last_3`
3. **Categorical feature**: `constructor_tier`

In [1]:
import os
import fastf1
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

RANDOM_SEED = 414
np.random.seed(RANDOM_SEED)

## 1. Load Validation and Train Data
Using the same temporal boundary logic from Lab 1.

In [2]:
if not os.path.exists('f1_cache'):
    os.makedirs('f1_cache')
fastf1.Cache.enable_cache('f1_cache')

def get_f1_data(seasons):
    all_results = []
    for year in seasons:
        schedule = fastf1.get_event_schedule(year)
        races = schedule[schedule['EventFormat'] != 'testing']
        for index, row in races.iterrows():
            gp_name = row['EventName']
            round_num = row['RoundNumber']
            try:
                session = fastf1.get_session(year, round_num, 'R')
                session.load(laps=False, telemetry=False, weather=False)
                results = session.results
                df_gp = results[['DriverNumber', 'FullName', 'Abbreviation', 'TeamName', 
                                 'GridPosition', 'Position', 'Status', 'Points']].copy()
                df_gp['Season'] = year
                df_gp['Round'] = round_num
                df_gp['GPName'] = gp_name
                df_gp['IsTop10'] = (df_gp['Position'] <= 10).astype(int)
                all_results.append(df_gp)
            except:
                pass
    return pd.concat(all_results, ignore_index=True)

df_f1 = get_f1_data([2022, 2023, 2024])
df_f1['GridPosition'] = df_f1['GridPosition'].fillna(20)
df_f1.loc[df_f1['GridPosition'] == 0, 'GridPosition'] = 20
print('Data Loaded!')

core           INFO 	Loading data for Bahrain Grand Prix - Race [v3.8.1]
req            INFO 	No cached data found for session_info. Loading data...
_api           INFO 	Fetching session info data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for driver_info. Loading data...
_api           INFO 	Fetching driver list...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for race_control_messages. Loading data...
_api           INFO 	Fetching race control messages...
req            INFO 	Data has been written to cache!
core           INFO 	Finished loading data for 20 drivers: ['16', '55', '44', '63', '20', '77', '31', '22', '14', '24', '47', '18', '23', '3', '4', '6', '27', '11', '1', '10']
core           INFO 	Loading data for Saudi Arabian Grand Prix - Race [v3.8.1]
req            INFO 	No cached data found for session_info. Loading data...
_api           INFO 	Fetching session info data...

Data Loaded!


## 2. Feature Engineering

In [3]:
df_f1.sort_values(by=['Season', 'Round'], inplace=True)

# Feature 1: Lag feature (prev_race_position)
# Leakage guard applied: using shift(1)
df_f1['prev_race_position'] = df_f1.groupby('DriverNumber')['Position'].shift(1)
df_f1['prev_race_position'] = df_f1['prev_race_position'].fillna(20)

# Feature 2: Rolling aggregate (avg_position_last_3)
# Leakage guard applied: using shift(1)
df_f1['avg_position_last_3'] = df_f1.groupby('DriverNumber')['Position'].transform(
    lambda x: x.rolling(3, min_periods=1).mean().shift(1)
)
df_f1['avg_position_last_3'] = df_f1['avg_position_last_3'].fillna(20)

# Feature 3: Categorical feature (constructor_tier)
top_teams = ['Red Bull Racing', 'Ferrari', 'Mercedes']
mid_teams = ['McLaren', 'Aston Martin', 'Alpine']
def get_tier(team):
    if team in top_teams: return 1
    elif team in mid_teams: return 2
    else: return 3
df_f1['constructor_tier'] = df_f1['TeamName'].apply(get_tier)

## 3. Temporal Validation Split & Logistic Regression

In [4]:
train_df = df_f1[((df_f1['Season'] == 2022) | (df_f1['Season'] == 2023))].copy()
val_df = df_f1[(df_f1['Season'] == 2024) & (df_f1['Round'] <= 12)].copy()

features = ['GridPosition', 'prev_race_position', 'avg_position_last_3', 'constructor_tier']
target = 'IsTop10'

X_train, y_train = train_df[features], train_df[target]
X_val, y_val = val_df[features], val_df[target]

model = LogisticRegression(random_state=RANDOM_SEED)
model.fit(X_train, y_train)
y_pred = model.predict(X_val)
y_prob = model.predict_proba(X_val)[:, 1]

## 4. Evaluation Metrics

In [5]:
acc = accuracy_score(y_val, y_pred)
prec = precision_score(y_val, y_pred)
rec = recall_score(y_val, y_pred)
f1 = f1_score(y_val, y_pred)
auc = roc_auc_score(y_val, y_prob)

print(f"Accuracy: {acc:.4f}")
print(f"Precision: {prec:.4f}")
print(f"Recall: {rec:.4f}")
print(f"F1: {f1:.4f}")
print(f"ROC-AUC: {auc:.4f}")

Accuracy: 0.8285
Precision: 0.8435
Recall: 0.8083
F1: 0.8255
ROC-AUC: 0.8984


## 5. Error Analysis

In [6]:
val_df['prediction'] = y_pred
errors = val_df[val_df['IsTop10'] != val_df['prediction']]

fp = errors[(errors['prediction'] == 1) & (errors['IsTop10'] == 0)]
fn = errors[(errors['prediction'] == 0) & (errors['IsTop10'] == 1)]

print("Top False Positives (Predicted Top-10, Finished outside):\n", fp['FullName'].value_counts().head(3))
print("\nTop False Negatives (Predicted outside, Finished Top-10):\n", fn['FullName'].value_counts().head(3))

Top False Positives (Predicted Top-10, Finished outside):
 FullName
Lance Stroll       3
Charles Leclerc    3
George Russell     2
Name: count, dtype: int64

Top False Negatives (Predicted outside, Finished Top-10):
 FullName
Yuki Tsunoda       6
Nico Hulkenberg    5
Kevin Magnussen    2
Name: count, dtype: int64
